
# 📰 Fake News Detection - Unified Model

This notebook consolidates multiple models to detect fake news using NLP techniques. It loads the dataset, processes the text, and trains a transformer-based classification model.

## 📁 Dataset Files
- `Train.csv`: Contains labeled training data.
- `Test.csv`: Contains text data to predict labels.
- `sample_submission.csv`: Format for submitting predictions.

## 📌 Libraries Required
- pandas, numpy
- sklearn
- transformers
- torch

Make sure the dataset is in the same directory as this notebook or adjust the paths accordingly.


In [9]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

import warnings
warnings.filterwarnings("ignore")


In [10]:

train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')

train_df.dropna(subset=['text'], inplace=True)
print("Training Data Shape:", train_df.shape)
train_df.head()


Training Data Shape: (10, 2)


,text,label
0,The president announced a new policy today.,0
1,Aliens have landed in New York and taken over ...,1
2,Scientists discovered a new species of bird in...,0
3,"The moon is made of cheese, claims anonymous s...",1
4,"Vaccines cause autism, according to unverified...",1


In [11]:

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

from datasets import Dataset

dataset = Dataset.from_pandas(train_df[['text', 'label']])
dataset = dataset.train_test_split(test_size=0.1)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['text'])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")


Map: 100%|██████████| 1/1 [00:00<00:00, 156.40 examples/s]


In [ ]:

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:

preds = trainer.predict(tokenized_datasets["test"])
y_pred = np.argmax(preds.predictions, axis=-1)
y_true = preds.label_ids

print("Classification Report:\n", classification_report(y_true, y_pred))
print("Accuracy Score:", accuracy_score(y_true, y_pred))
